# To convert drive miniseeds to currently used format + rollback script
## network.station.location.channel_YYYYMMDDhhmmss.mseed
### e.g. ZE.2311..GPZ_20230715060000.mseed

In [2]:
### ===== FULL COMPLETE RUN ===== ###

#### shutil.move() instead of shutil.copy2() ####

import os
import shutil
import csv
from obspy import UTCDateTime
from collections import defaultdict
from datetime import timedelta

# ===== BEFORE RUNNING ===== #
# make sure correct mseed folders inputted, and CHANGE CSV FILE NAME TO AVOID OVERWRITING
# test one or a few mseeds first by setting:
#                                           TEST_ONE_FILE = True
#                                           TEST_LIMIT = 20


# ===== GOAL =====
print("Rename mseeds to format: 'network.station.location.channel_YYYYMMDDhhmmss.mseed'")

# folder "D:\Nisqually_2024_MSEED" contains 1,950 files
# folder "D:\2024_MSEED_From_PI" contains 2,958 files + 33
sn_to_sta_24 = {
    "453019301": "2401",
    "453011911": "2402",
    "453011739": "2403",
    "453005685": "2404",
    "453012150": "2405",
    "453013918": "2406",
    "453013539": "2407",
    "453019813": "2408",
    "453005807": "2409",
    "453005654": "2410",
    "453014184": "2411",
    "453017424": "2412",
    "453012390": "2413",
    "453013376": "2414",
    "453012976": "2415",
    "453019821": "2416"
}

# folder "D:\Nisqually_2023_MSEED" contains 3,882 files
sn_to_sta_23 = {
    "1029438": "2301",
    "1029414": "2302",
    "1029458": "2303",
    "1029404": "2304",
    "1029497": "2305",
    "1029468": "2306",
    "1029496": "2307",
    "1029442": "2308",
    "1029427": "2309",
    "1029483": "2310",
    "1029410": "2311",
    "1029428": "2312",
    "1029462": "2313",
    "1029411": "2314",
    "1029444": "2315",
    "1029273": "2316",
}

# folder "D:\2025_MSEED_From_PI" contains 4,614 files
# folder "D:\2025_MSEED_From_Nodes" contains 3,696 files
sn_to_sta_25 = {
    "453013885": "2501",
    "453012461": "2502",
    "453019222": "2503",
    "453014233": "2504",
    "453011844": "2505",
    "453019889": "2506",
    "453005790": "2507",
    "453013098": "2508",
    "453013851": "2509",
    "453012498": "2510",
    "453018943": "2511",
    "453005768": "2512",
    "453020720": "2513",
    "453012164": "2514",
    "453012536": "2515",
    "453012683": "2516",
    "453013322": "NHR1", # high-rate transect, 10m
    "453019414": "NHR2", # high-rate transect, 25m
    "453013783": "NHR3", # high-rate transect, 40m
    "453020406": "NHR4", # high-rate transect, 60m
    "453013230": "NHR5", # high-rate, transect, 80m
    "453013100": "NHRC" # 10.1m from NHR1, 0.7m from river edge
    ## calibration node handle separately, eg for t in timeframe: C2501,...NHRC
}

sn_to_calib_time = {
    "453005764": [
        ("202509251951", "20250925201456", "C2501"),
        ("202509251823", "20250925185433", "C2502"),
        ("202509182029", "20250918204446", "C2503"),
        ("202509181828", "20250918184210", "C2504"),
        ("202509181652", "20250918171641", "C2505"),
        ("202509281752", "20250928181752", "C2506"),
        ("202509282005", "20250928202541", "C2507"),
        ("202509301835", "20250930185930", "C2508"),
        ("202509282239", "20250928230936", "C2509"),
        ("202510232110", "20251023212626", "C2510"),
        ("202510232153", "20251023223832", "C2511"),
        ("202510232331", "20251024002848", "C2512"),
        ("202510230057", "20251024011239", "C2513"),
        ("202509232144", "20250923221944", "C2515"),
        ("202509240020", "20250924003831", "C2516")
    ]
}

# ===== TEST MODE =====
TEST_ONE_FILE = False
TEST_LIMIT = None # or 0 for all #20
TEST_FILENAME = "453019821.0049.2024.07.24.00.00.00.000.N.miniseed"  

# ===== CONFIG =====
network = "ZE"
location = ""

channel_map = {
    "Z": "GPZ",
    "N": "GP1",
    "E": "GP2"
}

# calibration settings - adding margin in case of slight ON/OFF time discrepancies
start_margin = 20 # +/- 20s of start listed
end_plus = 15*60 # 15 mins added to end

# ===== DICTIONARIES =====
sn_to_sta_23 = sn_to_sta_23
sn_to_sta_24 = sn_to_sta_24
sn_to_sta_25 = sn_to_sta_25

sn_to_calib_time = sn_to_calib_time

# ===== LOGGING SETUP =====
log_path = r"D:\rollback_rename_logs\NAME_THE_CSV.csv"      ##########
os.makedirs(os.path.dirname(log_path), exist_ok=True)
log_file = open(log_path, "w", newline="")
log_writer = csv.writer(log_file)
log_writer.writerow(["original_path", "new_path"])


def process_station_dict(station_dict):  # could improve this by adding source_root, target_root, csv file name as arguments
    """
    Renames miniseed files according to the type of dictionary:
    - calibration dict (serial -> [(start, end, station), ...])
      applies time windows using obspy UTCDateTime
    - regular dict (serial -> station)
      direct mapping
    """
    
    # determine source/target paths based on dict type
    # calibration dict will have its own folder
    if station_dict == sn_to_calib_time:      ################ CHANGE THESE WHEN USING CALIBRATION DATA  ######################
        source_root = r"D:\calibration_test"  # or D"\2025_MSEED_From_Nodes\calibration (2 copies of calib pre-rename) 
        target_root = r"D:\calibration_rename_test2"
        is_calibration = True

    # regular node dict will have own folder based on year
    elif station_dict in [sn_to_sta_23, sn_to_sta_24, sn_to_sta_25]:

        if station_dict is sn_to_sta_23: # don't really need this section anymore, changed path saving
            year = "2023"
        elif station_dict is sn_to_sta_24:
            year = "2024"
        elif station_dict is sn_to_sta_25:
            year = "2025"

        source_root = r"D:\2025_MSEED_4000sps"    ###### WHAT DATA?
        #target_path = rf"{year}\2025_MSEED_4000sps"
        target_root = f"{source_root}_rename"

        is_calibration = False

    else:
        print("[ERROR] Unknown station dictionary provided")
        return
        
    # helper for parsing calibration periods
    calib_periods = []
    if is_calibration:
        for sn, entries in station_dict.items():
            for start_str, end_str, sta in entries:
                start_dt = UTCDateTime(start_str) - start_margin # margin to catch notetaking discrepancies
                end_dt = UTCDateTime(end_str) + end_plus # end buffer since official OFF times unknown (just have last impact times)
                calib_periods.append((sn, sta, start_dt, end_dt))

    station_summary = defaultdict(list)

    processed_count = 0

    # walk through files
    for root, dirs, files in os.walk(source_root): # loops through folders in source path
        for file in files:

            if not file.endswith(".miniseed"): # only process miniseeds
                continue

            # ===== TEST MODE FILTER =====
            # this will "exit" test mode when TEST_ONE_FILE = False and TEST_LIMIT = None
            if TEST_ONE_FILE and file != TEST_FILENAME:
                continue

            parts = file.split(".") # separate parts of orginal file naming schema

            try:
                serial = parts[0]
                year, month, day = parts[2], parts[3], parts[4]
                hour, minute, second = parts[5], parts[6], parts[7]
                direction = parts[9]

                file_ts = UTCDateTime(f"{year}{month}{day}{hour}{minute}{second}")

                if direction not in channel_map:
                    print(f"[WARN] Unknown direction {direction} in {file}")
                    continue

                channel = channel_map[direction]

                # ===== CALIBRATION =====
                if is_calibration:
                    station = None
                    for sn, sta, start, end in calib_periods:
                        if serial == sn and start <= file_ts <= end:
                            station = sta
                            break

                    if not station:
                        print(f"[WARN] {file} not in calibration period")
                        continue

                # ===== REGULAR =====
                else:
                    if serial in station_dict:
                        station = station_dict[serial]
                    else:
                        print(f"[SKIP] Unknown serial {serial}")
                        continue

                timestamp_str = file_ts.strftime("%Y%m%d%H%M%S")
                new_filename = f"{network}.{station}.{location}.{channel}_{timestamp_str}.mseed"

                dest_dir = os.path.join(target_root, f"NO{station}", channel)
                os.makedirs(dest_dir, exist_ok=True)

                src_path = os.path.join(root, file)
                dest_path = os.path.join(dest_dir, new_filename)

                # ===== COLLISION PROTECTION =====
                if os.path.exists(dest_path):
                    print(f"[COLLISION] {dest_path} exists, adding suffix")
                    base, ext = os.path.splitext(dest_path)
                    counter = 1
                    while os.path.exists(dest_path):
                        dest_path = f"{base}_{counter}{ext}"
                        counter += 1

                # ===== MOVE (FINAL STEP) =====
                shutil.move(src_path, dest_path)
                # test with shutil.copy2() before permanently moving renaming/moving

                print(f"[MOVED] {file} → {dest_path}")

                # ===== LOGGING =====
                log_writer.writerow([src_path, dest_path])

                station_summary[station].append(file_ts)

                # TESTING 20 FILES
                processed_count += 1

                if TEST_LIMIT is not None and processed_count >= TEST_LIMIT:
                #if TEST_LIMIT and processed_count >= TEST_LIMIT:
                    print(f"\n[TEST LIMIT] Reached {TEST_LIMIT} files. Stopping.\n")
                    return

                # ===== STOP AFTER ONE FILE =====
                if TEST_ONE_FILE:
                    print("\n[TEST MODE] Processed one file. Stopping.\n")
                    return

            except Exception as e:
                print(f"[ERROR] Processing {file}: {e}")

    print("\n=== STATION SUMMARY ===")
    for sta, times in station_summary.items():
        print(f"{sta}: {len(times)} files")

    print("\nDone\n")


# ===== RUN =====
process_station_dict(sn_to_sta_25)

# if calibration:
#process_station_dict(sn_to_calib_time)

# ===== CLOSE LOG =====
log_file.close()

Rename mseeds to format: 'network.station.location.channel_YYYYMMDDhhmmss.mseed'
[MOVED] 453013100.0001.2025.10.21.22.52.08.000.E.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GP2\ZE.NHRC..GP2_20251021225208.mseed
[MOVED] 453013100.0001.2025.10.21.22.52.08.000.N.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GP1\ZE.NHRC..GP1_20251021225208.mseed
[MOVED] 453013100.0001.2025.10.21.22.52.08.000.Z.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GPZ\ZE.NHRC..GPZ_20251021225208.mseed
[MOVED] 453013100.0002.2025.10.22.00.00.00.000.E.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GP2\ZE.NHRC..GP2_20251022000000.mseed
[MOVED] 453013100.0002.2025.10.22.00.00.00.000.N.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GP1\ZE.NHRC..GP1_20251022000000.mseed
[MOVED] 453013100.0002.2025.10.22.00.00.00.000.Z.miniseed → D:\2025_MSEED_4000sps_rename\NONHRC\GPZ\ZE.NHRC..GPZ_20251022000000.mseed
[MOVED] 453013230.0001.2025.10.07.22.38.14.000.E.miniseed → D:\2025_MSEED_4000sps_rename\NONHR5\GP2\ZE.NHR5..GP2_20

# Rollback/Undo Script
## for if the orignal drive mseed naming format is needed

In [7]:
### rollback script aka UNDO if needed

import csv
import os
import shutil

log_path = r"D:\rollback_rename_logs\Nisqually_2023_MSEED.csv"
# log_path = r"D:\rollback_rename_logs\2024_MSEED_From_PI.csv"
# log_path = r"D:\rollback_rename_logs\Nisqually_2024_MSEED.csv"
# log_path = r"D:\rollback_rename_logs\2025_MSEED_From_PI.csv"
# log_path = r"D:\rollback_rename_logs\2025_MSEED_From_nodes.csv"
# log_path = r"D:\rollback_rename_logs\2025_MSEED_4000sps.csv"

print(f"[ROLLBACK] Using log: {log_path}")

if not os.path.exists(log_path):
    raise FileNotFoundError("Log file not found.")

with open(log_path, "r", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        original = row["original_path"]
        moved = row["new_path"]

        try:
            # only rollback if moved file exists
            if os.path.exists(moved):
                os.makedirs(os.path.dirname(original), exist_ok=True)
                shutil.move(moved, original)
                print(f"[RESTORED] {moved} → {original}")
            else:
                print(f"[MISSING] {moved} not found, skipping")

        except Exception as e:
            print(f"[ERROR] {moved}: {e}")

print("\n[ROLLBACK COMPLETE]")

[ROLLBACK] Using log: D:\rename_test_log.csv
[RESTORED] D:\2024_mseed_rename_new\NO2416\GP1\ZE.2416..GP1_20240724000000.mseed → D:\2024_MSEED_From_PI_copy10\453019821.0049.2024.07.24.00.00.00.000.N.miniseed

[ROLLBACK COMPLETE]


In [ ]:
# older version, but also works for calib and regular nodes

### newest trial -- full process
# works for calibration
# checking regular nodes - worked for NO2416
import os
import shutil
from obspy import UTCDateTime
from collections import defaultdict
from datetime import timedelta

# ===== GOAL =====
print("Rename mseeds to format: 'network.station.location.channel_YYYYMMDDhhmmss.mseed'")

# ===== CONFIG =====
network = "ZE"
location = ""  # keep empty unless want something else

# channel mapping
channel_map = {
    "Z": "GPZ",
    "N": "GP1",
    "E": "GP2"
}

# calibration settings - adding margin in case of slight ON/OFF time discrepancies
start_margin = 20 # +/- 20s of start listed
end_plus = 15*60 # 15 mins added to end

# ===== DICTIONARIES =====
# regular station dictionaries (serial --> station)
sn_to_sta_23 = sn_to_sta_23
sn_to_sta_24 = sn_to_sta_24
sn_to_sta_25 = sn_to_sta_25

# calibration dictionary format: serial --> [(start, end, station), ...]
# start/end = YYYYMMDDHHMM (start), YYYYMMDDHHMMSS (last impact)
sn_to_calib_time = sn_to_calib_time
# sn_to_calib_time = {
#     "453005764": [
#         ("202509181652", "20250918171641", "C2505"),
#         ("202509181828", "20250918184210", "C2504"),
#         ("202509182029", "20250918204446", "C2503"),
#         ("202509232144", "20250923221944", "C2515"),
#         ("202509230020", "20250923003831", "C2516"),
#         ("202509251823", "20250925185433", "C2502"),
#         ("202509251951", "20250925201456", "C2501"),
#         ("202509281752", "20250928181752", "C2506"),
#         ("202509282005", "20250928202541", "C2507"),
#         ("202509282239", "20250928230936", "C2509"),
#         ("202509301835", "20250930185930", "C2508"),
#         ("202510232110", "20251023212626", "C2510"),
#         ("202510232153", "20251023223832", "C2511"),
#         ("202510232331", "20251024002848", "C2512"),
#         ("202510230057", "20251024011239", "C2513")
#     ]
# }

# ===== FUNCTION TO PROCESS ANY DICTIONARY =====
def process_station_dict(station_dict):
    """
    Renames miniseed files according to the type of dictionary:
    - calibration dict (serial -> [(start, end, station), ...])
      applies time windows using obspy UTCDateTime
    - regular dict (serial -> station)
      direct mapping
    """
    
    # determine source/target paths based on dict type
    # calibration dict will have its own folder
    if station_dict == sn_to_calib_time:
        source_root = r"D:\calibration_test"
        target_root = r"D:\calibration_rename_test"
        is_calibration = True
    # regular node dict will have own folder based on year
    elif station_dict in [sn_to_sta_23, sn_to_sta_24, sn_to_sta_25]:

        if station_dict is sn_to_sta_23:
            year = "2023"
        elif station_dict is sn_to_sta_24:
            year = "2024"
        elif station_dict is sn_to_sta_25:
            year = "2025"

        # source_root = rf"D:\{year}_MSEED_From_PI_copy10"
        # target_root = rf"D:\{year}_convert_test\10_copies"

        source_root = f"D:\2024_MSEED_From_PI"
        taret_root = rf"D:\{year}_mseed_rename"
    
        is_calibration = False
        
    # elif station_dict in [sn_to_sta_23, sn_to_sta_24, sn_to_sta_25]:
    #     year_map = {sn_to_sta_23:"2023", sn_to_sta_24:"2024", sn_to_sta_25:"2025"}
    #     source_root = rf"D:\2024_MSEED_From_PI_copy10"
    #     #source_root = rf"D:\{year_map[station_dict]}_mseed_test"
    #     target_root = rf"D:\2024_convert_test\10_copies"
    #     #target_root = rf"D:\{year_map[station_dict]}_mseed_rename_test"
    #     is_calibration = False
    else:
        print("[ERROR] Unknown station dictionary provided")
        return

    # helper for parsing calibration periods
    calib_periods = []
    if is_calibration:
        for sn, entries in station_dict.items():
            for start_str, end_str, sta in entries:
                # convert to obspy UTCDateTime object
                start_dt = UTCDateTime(start_str) - start_margin # margin to catch notetaking discrepancies
                end_dt = UTCDateTime(end_str) + end_plus # end buffer since official OFF times unknown (just have last impact times)
                calib_periods.append((sn, sta, start_dt, end_dt))

    # dictionary to summarize assignments
    station_summary = defaultdict(list)

    # walk through files
    for root, dirs, files in os.walk(source_root): # loops through folders in source path
        for file in files:
            if not file.endswith(".miniseed"): # only process miniseeds
                continue
            parts = file.split(".") # separate parts of orginal file naming schema
            try:
                serial = parts[0]
                year, month, day = parts[2], parts[3], parts[4]
                hour, minute, second = parts[5], parts[6], parts[7]
                direction = parts[9]

                # convert to UTCDateTime for new file name
                file_ts = UTCDateTime(f"{year}{month}{day}{hour}{minute}{second}")

                # map channel
                if direction not in channel_map:
                    print(f"[WARN] Unknown direction {direction} in {file}")
                    continue
                channel = channel_map[direction]

                # --- calibration logic ---
                if is_calibration:
                    station = None
                    for sn, sta, start, end in calib_periods:
                        # DEBUG: show what window is being checked
                        print(f"[DEBUG CHECK] File {file_ts} | Checking {sta} window: {start} → {end}")

                        if serial == sn and start <= file_ts <= end: # change to < end if worried about off/on time overlap
                            # DEBUG: show MATCH
                            print(f"[DEBUG MATCH] File {file_ts} MATCHED {sta}")
                            print(f"              Window: {start} → {end}")
                            # print(f"\n[DEBUG MATCH]")
                            # print(f"File time:   {file_ts}")
                            # print(f"Station:     {sta}")
                            # print(f"Window:      {start} → {end}")
                            station = sta # assign station if within time period
                            break # file should only belong to one station
                            
                    if not station: # if not in time period --> skip file
                        print(f"[WARN] {file} does NOT fall in any calibration period")
                        continue
                # --- regular node logic ---
                else:
                    if serial in station_dict:
                        station = station_dict[serial]
                    else:
                        print(f"[SKIP] Unknown serial {serial} in {file}")
                        continue

                # build new filename
                timestamp_str = file_ts.strftime("%Y%m%d%H%M%S")
                new_filename = f"{network}.{station}.{location}.{channel}_{timestamp_str}.mseed"

                # build destination path
                dest_dir = os.path.join(target_root, f"NO{station}", channel)
                os.makedirs(dest_dir, exist_ok=True)
                src_path = os.path.join(root, file)
                dest_path = os.path.join(dest_dir, new_filename)

                # copy files (change to .move() when sure)
                shutil.copy2(src_path, dest_path)
                print(f"[ASSIGNED] {file} → {dest_path}")
                station_summary[station].append(file_ts)

            except Exception as e:
                print(f"[ERROR] Processing {file}: {e}")

    # print summary of changes
    print("\n=== STATION SUMMARY ===")
    for sta, times in station_summary.items():
        min_time = min(times)
        max_time = max(times)
        print(f"Station {sta}: {len(times)} files, from {min_time} to {max_time}")
    print("\nDone\n")


## call by

# if calibration
#process_station_dict(sn_to_calib_time)

# if regular nodes (example for 2025)
process_station_dict(sn_to_sta_24)